In [5]:
pip install gymnasium gymnasium-robotics mujoco

Note: you may need to restart the kernel to use updated packages.


In [6]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pickle
import os
import random
import time
from collections import deque

import gymnasium as gym
import gymnasium_robotics


SEED = 42
np.random.seed(SEED)
random.seed(SEED)


_probe = gym.make("FetchReach-v4")
_obs, _ = _probe.reset(seed=SEED)

OBS_DIM      = _obs["observation"].shape[0]          
GOAL_DIM     = _obs["desired_goal"].shape[0]         
ACTION_DIM   = _probe.action_space.shape[0]          
ACTION_BOUND = float(_probe.action_space.high[0])    
MAX_STEPS    = _probe.spec.max_episode_steps         
DIST_THRESH  = _probe.unwrapped.distance_threshold   
_probe.close()

IN_DIM = OBS_DIM + GOAL_DIM                          

print(f"obs_dim={OBS_DIM}  goal_dim={GOAL_DIM}  action_dim={ACTION_DIM}")
print(f"action_bound={ACTION_BOUND}  max_steps={MAX_STEPS}  dist_threshold={DIST_THRESH} m")
print(f"Влезна димензија за мрежите (obs+goal): {IN_DIM}")

obs_dim=10  goal_dim=3  action_dim=4
action_bound=1.0  max_steps=50  dist_threshold=0.05 m
Влезна димензија за мрежите (obs+goal): 13


In [7]:

def relu(x):    
    return np.maximum(0.0, x)

def relu_g(x):  
    return (x > 0.0).astype(float)

def tanh_g(x):  
    return 1.0 - np.tanh(x)**2

def he(fan_in, fan_out):
    return np.random.randn(fan_in, fan_out) * np.sqrt(2.0 / fan_in)

def uni(fan_in, fan_out, lim=3e-3):
    return np.random.uniform(-lim, lim, (fan_in, fan_out))

# ── Layer Normalization и прецизен Backprop ──────────────────
def layer_norm(x, g, b, eps=1e-5):
    mu = x.mean(-1, keepdims=True)
    var = x.var(-1, keepdims=True)
    std = np.sqrt(var + eps)
    return g * (x - mu) / std + b

def layer_norm_grad(x, g, b, dout, eps=1e-5):
    """Математички точен backprop низ LayerNorm по последната оска D."""
    B, D = x.shape
    mu = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    std = np.sqrt(var + eps)
    x_hat = (x - mu) / std
    
    dg_out = (dout * x_hat).sum(axis=0)
    db_out = dout.sum(axis=0)
    
    dx_hat = dout * g
    # Компактна аналитичка форма на извод на LN по карактеристики
    dx = (1.0 / D) * (1.0 / std) * (
        D * dx_hat - dx_hat.sum(axis=-1, keepdims=True) - x_hat * (dx_hat * x_hat).sum(axis=-1, keepdims=True)
    )
    return dx, dg_out, db_out



In [8]:
H1, H2 = 400, 300  


class Actor:
    def __init__(self):
        self.W1 = he(IN_DIM, H1);          self.b1 = np.zeros((1, H1))
        self.g1 = np.ones(H1);             self.d1 = np.zeros(H1)

        self.W2 = he(H1, H2);              self.b2 = np.zeros((1, H2))
        self.g2 = np.ones(H2);             self.d2 = np.zeros(H2)

        self.W3 = uni(H2, ACTION_DIM);     self.b3 = np.zeros((1, ACTION_DIM))
        self._c = {}

    def params(self):
        return [self.W1, self.b1, self.g1, self.d1,
                self.W2, self.b2, self.g2, self.d2,
                self.W3, self.b3]

    def set_params(self, p):
        (self.W1, self.b1, self.g1, self.d1,
         self.W2, self.b2, self.g2, self.d2,
         self.W3, self.b3) = p

    def param_dict(self):
        return {"W1": self.W1, "b1": self.b1, "g1": self.g1, "d1": self.d1,
                "W2": self.W2, "b2": self.b2, "g2": self.g2, "d2": self.d2,
                "W3": self.W3, "b3": self.b3}

    def set_param_dict(self, d):
        for k, v in d.items():
            setattr(self, k, v)

    def forward(self, x, store=False):
        z1  = x @ self.W1 + self.b1
        n1  = layer_norm(z1, self.g1, self.d1)
        a1  = relu(n1)

        z2  = a1 @ self.W2 + self.b2
        n2  = layer_norm(z2, self.g2, self.d2)
        a2  = relu(n2)

        z3  = a2 @ self.W3 + self.b3
        out = np.tanh(z3) * ACTION_BOUND

        if store:
            self._c = dict(x=x, z1=z1, n1=n1, a1=a1, z2=z2, n2=n2, a2=a2, z3=z3, out=out)
        return out

    def backward(self, dout):
        c = self._c
        dz3 = dout * tanh_g(c["z3"]) * ACTION_BOUND
        dW3 = c["a2"].T @ dz3
        db3 = dz3.sum(0, keepdims=True)

        da2 = dz3 @ self.W3.T
        dn2 = da2 * relu_g(c["n2"])
        dx2, dg2, dd2 = layer_norm_grad(c["z2"], self.g2, self.d2, dn2)
        dW2 = c["a1"].T @ dx2
        db2 = dx2.sum(0, keepdims=True)

        da1 = dx2 @ self.W2.T
        dn1 = da1 * relu_g(c["n1"])
        dx1, dg1, dd1 = layer_norm_grad(c["z1"], self.g1, self.d1, dn1)
        dW1 = c["x"].T @ dx1
        db1 = dx1.sum(0, keepdims=True)

        return {"W1": dW1, "b1": db1, "g1": dg1, "d1": dd1,
                "W2": dW2, "b2": db2, "g2": dg2, "d2": dd2,
                "W3": dW3, "b3": db3}



class Critic:
    def __init__(self):
        self.W1s = he(IN_DIM, H1);         self.b1 = np.zeros((1, H1))
        self.g1  = np.ones(H1);            self.d1 = np.zeros(H1)

        self.W2  = he(H1 + ACTION_DIM, H2); self.b2 = np.zeros((1, H2))
        self.g2  = np.ones(H2);             self.d2 = np.zeros(H2)

        self.W3  = uni(H2, 1);             self.b3 = np.zeros((1, 1))
        self._c  = {}

    def params(self):
        return [self.W1s, self.b1, self.g1, self.d1,
                self.W2,  self.b2, self.g2, self.d2,
                self.W3,  self.b3]

    def set_params(self, p):
        (self.W1s, self.b1, self.g1, self.d1,
         self.W2,  self.b2, self.g2, self.d2,
         self.W3,  self.b3) = p

    def param_dict(self):
        return {"W1s": self.W1s, "b1": self.b1, "g1": self.g1, "d1": self.d1,
                "W2":  self.W2,  "b2": self.b2, "g2": self.g2, "d2": self.d2,
                "W3":  self.W3,  "b3": self.b3}

    def set_param_dict(self, d):
        for k, v in d.items():
            setattr(self, k, v)

    def forward(self, sg, a, store=False):
        z1  = sg @ self.W1s + self.b1
        n1  = layer_norm(z1, self.g1, self.d1)
        a1  = relu(n1)

        za  = np.concatenate([a1, a], axis=1) 
        z2  = za @ self.W2 + self.b2
        n2  = layer_norm(z2, self.g2, self.d2)
        a2  = relu(n2)

        Q   = a2 @ self.W3 + self.b3

        if store:
            self._c = dict(sg=sg, a=a, z1=z1, n1=n1, a1=a1, za=za, z2=z2, n2=n2, a2=a2, Q=Q)
        return Q

    def backward(self, dQ):
        c = self._c
        dW3 = c["a2"].T @ dQ
        db3 = dQ.sum(0, keepdims=True)
        da2 = dQ @ self.W3.T

        dn2 = da2 * relu_g(c["n2"])
        dx2, dg2, dd2 = layer_norm_grad(c["z2"], self.g2, self.d2, dn2)
        dza = dx2 @ self.W2.T
        dW2 = c["za"].T @ dx2
        db2 = dx2.sum(0, keepdims=True)

        dJ_da = dza[:, H1:]     # Градиент на критикот во однос на акцијата
        da1p  = dza[:, :H1]     # Градиент кон состојбата

        dn1 = da1p * relu_g(c["n1"])
        dx1, dg1, dd1 = layer_norm_grad(c["z1"], self.g1, self.d1, dn1)
        dW1s = c["sg"].T @ dx1
        db1  = dx1.sum(0, keepdims=True)

        grads = {"W1s": dW1s, "b1": db1, "g1": dg1, "d1": dd1,
                 "W2":  dW2,  "b2": db2, "g2": dg2, "d2": dd2,
                 "W3":  dW3,  "b3": db3}
        return grads, dJ_da



In [9]:

class Adam:
    def __init__(self, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8, clip=5.0):
        self.lr, self.b1, self.b2, self.eps = lr, beta1, beta2, eps
        self.clip = clip
        self.m, self.v, self.t = {}, {}, 0

    def step(self, param_d, grad_d):
        self.t += 1
        out = {}
        for k, p in param_d.items():
            g = np.clip(grad_d[k], -self.clip, self.clip)
            self.m.setdefault(k, np.zeros_like(p))
            self.v.setdefault(k, np.zeros_like(p))
            
            self.m[k] = self.b1 * self.m[k] + (1 - self.b1) * g
            self.v[k] = self.b2 * self.v[k] + (1 - self.b2) * g**2
            
            mh = self.m[k] / (1 - self.b1**self.t)
            vh = self.v[k] / (1 - self.b2**self.t)
            out[k] = p - self.lr * mh / (np.sqrt(vh) + self.eps)
        return out

def soft_update(target, online, tau=0.005):
    return [tau * o + (1 - tau) * t for t, o in zip(target, online)]

class HERBuffer:
    def __init__(self, max_episodes=5000, k=4):
        self.k   = k
        self.buf = deque(maxlen=max_episodes)

    def store_episode(self, ep):
        self.buf.append(ep)

    def __len__(self):
        return sum(len(ep) for ep in self.buf)

    def sample(self, batch_size):
        all_eps = list(self.buf)
        s_b, a_b, r_b, s2_b, d_b = [], [], [], [], []

        while len(s_b) < batch_size:
            ep  = random.choice(all_eps)
            T   = len(ep)
            t   = random.randint(0, T - 1)
            obs, ag, dg, act, rew, obs2, ag2, done = ep[t]

            
            if random.random() < self.k / (self.k + 1.0):
                fut   = random.randint(t, T - 1)
                g_use = ep[fut][6] 
                rew   = 0.0 if np.linalg.norm(ag2 - g_use) < DIST_THRESH else -1.0
            else:
                g_use = dg

            sg  = np.concatenate([obs,  g_use], dtype=np.float32)
            sg2 = np.concatenate([obs2, g_use], dtype=np.float32)

            s_b.append(sg)
            a_b.append(act.astype(np.float32))
            r_b.append(rew)
            s2_b.append(sg2)
            d_b.append(done)

        return (np.array(s_b,  dtype=np.float32),
                np.array(a_b,  dtype=np.float32),
                np.array(r_b,  dtype=np.float32).reshape(-1, 1),
                np.array(s2_b, dtype=np.float32),
                np.array(d_b,  dtype=np.float32).reshape(-1, 1))



In [10]:
#LAMBDA_S  = 0.02    
#LAMBDA_A  = 0.005   
LAMBDA_S  = 0.001    
LAMBDA_A  = 0.001    
EPS_SING  = 1e-4    
GAMMA     = 0.98    
TAU       = 0.005   
BATCH     = 256     

def compute_J_critic(Q_pred, y, s_batch, a_batch):
    """Ја пресметува точната цена и нејзиниот извод со вклучени казнени членови."""
    B = Q_pred.shape[0]
    td = Q_pred - y
    J_bell = float(np.mean(td**2))

    v_ee = s_batch[:, 5:8]
    v2   = (v_ee**2).sum(-1) + EPS_SING
    J_sing = float(LAMBDA_S * (1.0 / v2).mean())

    J_smooth = float(LAMBDA_A * np.mean(a_batch**2))

    J_total = J_bell + J_sing + J_smooth

    dJ_dQ = 2.0 * td / B 

    return J_total, J_bell, J_sing, J_smooth, dJ_dQ

def update_step(actor, critic, t_actor, t_critic, opt_a, opt_c, buf):
    sb, ab, rb, s2b, db = buf.sample(BATCH)
    a2_tgt = t_actor.forward(s2b)
    Q_tgt  = t_critic.forward(s2b, a2_tgt)
    y      = rb + GAMMA * (1.0 - db) * Q_tgt

   
    Q_pred = critic.forward(sb, ab, store=True)
    J_critic, J_bell, J_sing, J_smooth, dJ_dQ = compute_J_critic(Q_pred, y, sb, ab)
    
    c_grads, _ = critic.backward(dJ_dQ)
    critic.set_param_dict(opt_c.step(critic.param_dict(), c_grads))

    
    a_pred = actor.forward(sb, store=True)
    Q_fa   = critic.forward(sb, a_pred, store=True)

    J_actor = float(-np.mean(Q_fa))
    
    
    dL_dQ   = -np.ones_like(Q_fa) / BATCH
    _, dQ_da = critic.backward(dL_dQ)
    dL_da   = dQ_da + 2.0 * LAMBDA_A * a_pred / BATCH

    a_grads = actor.backward(dL_da)
    actor.set_param_dict(opt_a.step(actor.param_dict(), a_grads))

    
    t_actor.set_params( soft_update(t_actor.params(),  actor.params(),  TAU))
    t_critic.set_params(soft_update(t_critic.params(), critic.params(), TAU))

    return J_critic, J_actor, J_bell, J_sing, J_smooth



In [11]:
def save_model(actor, critic, path):
    """Ги зачувува научените параметри на невронските мрежи."""
    with open(path, "wb") as f:
        pickle.dump({
            "actor": actor.param_dict(),
            "critic": critic.param_dict()
        }, f)
    print(f"Моделот е успешно зачуван на патека: {path}")

In [12]:
N_EPISODES      = 1200  
WARMUP_EP       = 100   
OPT_PER_EP      = 40    
NOISE_START     = 0.20  
NOISE_END       = 0.02  
LOG_EVERY       = 50
SAVE_PATH       = "fetchreach_her_dpg_pure_numpy.pkl"

def noise_std(ep):
    frac = min(ep / N_EPISODES, 1.0)
    return NOISE_START + frac * (NOISE_END - NOISE_START)

def train():
    env = gym.make("FetchReach-v4")
    env.action_space.seed(SEED)
    env.reset(seed=SEED)

    actor, critic   = Actor(), Critic()
    t_actor, t_critic = Actor(), Critic()
    t_actor.set_params( [p.copy() for p in actor.params()])
    t_critic.set_params([p.copy() for p in critic.params()])

    opt_a = Adam(lr=1e-3)
    opt_c = Adam(lr=1e-3)
    buf   = HERBuffer(max_episodes=5000, k=4)

    recent_success = deque(maxlen=100)
    t0 = time.time()

    for ep in range(1, N_EPISODES + 1):
        obs_d, _ = env.reset()
        obs = obs_d["observation"]
        ag  = obs_d["achieved_goal"]
        dg  = obs_d["desired_goal"]
        ns  = noise_std(ep)
        ep_trans = []
        ep_success = False

        for step in range(MAX_STEPS):
            sg = np.concatenate([obs, dg]).reshape(1, -1).astype(np.float32)

            if ep <= WARMUP_EP:
                act = env.action_space.sample().astype(np.float32)
            else:
                act = actor.forward(sg)[0].astype(np.float32)
                act = np.clip(
                    act + (ns * np.random.randn(ACTION_DIM)).astype(np.float32),
                    -ACTION_BOUND, ACTION_BOUND
                )

            obs_d2, rew, term, trunc, info = env.step(act)
            obs2 = obs_d2["observation"]
            ag2  = obs_d2["achieved_goal"]
            done = float(term or trunc)

            ep_trans.append((obs.copy(), ag.copy(), dg.copy(),
                             act.copy(), float(rew),
                             obs2.copy(), ag2.copy(), done))
            obs, ag = obs2, ag2
            
            if info.get("is_success", 0.0):
                ep_success = True
            if term or trunc:
                break

        buf.store_episode(ep_trans)
        recent_success.append(float(ep_success))
        J_c = J_a = J_b = J_s = J_sm = 0.0
        if ep > WARMUP_EP and len(buf) >= BATCH:
            for _ in range(OPT_PER_EP):
                J_c, J_a, J_b, J_s, J_sm = update_step(
                    actor, critic, t_actor, t_critic, opt_a, opt_c, buf
                )

        if ep % LOG_EVERY == 0:
            sr = float(np.mean(recent_success))
            elapsed = time.time() - t0
            print(f"Епизода {ep:4d}/{N_EPISODES} | Успешност (SR)={sr:.1%} | Шум={ns:.3f} | "
                  f"J_Crit={J_c:.4f} (Bell={J_b:.4f}, Sing={J_s:.4f}) | "
                  f"J_Act={J_a:.4f} | Време={elapsed:.1f}s")

    env.close()
    save_model(actor, critic, SAVE_PATH)
    return actor, critic


trained_actor, trained_critic = train()

Епизода   50/1200 | Успешност (SR)=18.0% | Шум=0.193 | J_Crit=0.0000 (Bell=0.0000, Sing=0.0000) | J_Act=0.0000 | Време=3.3s
Епизода  100/1200 | Успешност (SR)=14.0% | Шум=0.185 | J_Crit=0.0000 (Bell=0.0000, Sing=0.0000) | J_Act=0.0000 | Време=7.3s
Епизода  150/1200 | Успешност (SR)=18.0% | Шум=0.178 | J_Crit=3.1154 (Bell=0.4205, Sing=2.6944) | J_Act=3.1999 | Време=125.5s
Епизода  200/1200 | Успешност (SR)=63.0% | Шум=0.170 | J_Crit=3.1773 (Bell=0.0366, Sing=3.1402) | J_Act=1.8053 | Време=239.0s
Епизода  250/1200 | Успешност (SR)=100.0% | Шум=0.163 | J_Crit=3.7587 (Bell=0.0346, Sing=3.7237) | J_Act=0.7677 | Време=352.9s
Епизода  300/1200 | Успешност (SR)=100.0% | Шум=0.155 | J_Crit=4.2186 (Bell=0.2179, Sing=4.0003) | J_Act=0.4359 | Време=466.7s
Епизода  350/1200 | Успешност (SR)=100.0% | Шум=0.147 | J_Crit=4.3066 (Bell=0.0422, Sing=4.2641) | J_Act=0.0036 | Време=582.5s
Епизода  400/1200 | Успешност (SR)=100.0% | Шум=0.140 | J_Crit=4.4566 (Bell=0.0205, Sing=4.4358) | J_Act=0.1689 | Време

In [14]:
import gymnasium as gym
import gymnasium_robotics
import numpy as np
import pickle
import time


TEST_EPISODES = 5
SAVE_PATH = "fetchreach_her_dpg_pure_numpy.pkl"

env = gym.make("FetchReach-v4", render_mode="human")

try:
    
    eval_actor = trained_actor
    print("Успешно пронајден трениран Actor во локалната меморија.")
except NameError:
 
    print(f"Не е пронајден модел во меморијата. Вчитување од: {SAVE_PATH} ...")
    if not os.path.exists(SAVE_PATH):
        raise FileNotFoundError(f"Грешка: Не постои фајл '{SAVE_PATH}'. Прво мора да го истренираш моделот во Чекор 7.")
        
    with open(SAVE_PATH, "rb") as f:
        data = pickle.load(f)
    
    # Креираме нов објект и му ги вбризгуваме зачуваните тежини
    eval_actor = Actor()
    eval_actor.set_param_dict(data["actor"])
for ep in range(1, TEST_EPISODES + 1):
    obs_d, _ = env.reset()
    obs = obs_d["observation"]
    dg  = obs_d["desired_goal"]
    
    ep_reward = 0
    success = False
    
    # Секоја епизода трае максимум 50 чекори
    for step in range(50):
        sg = np.concatenate([obs, dg]).reshape(1, -1).astype(np.float32)
        
        act = eval_actor.forward(sg)[0]
        
       
        obs_d2, rew, term, trunc, info = env.step(act)
        
        obs = obs_d2["observation"]
        ep_reward += rew
        
        if info.get("is_success", 0.0):
            success = True
            
        
        time.sleep(0.04)
        
        if term or trunc:
            break
            
    status = "УСПЕШНО" if success else "НЕУСПЕШНО"
    print(f" Епизода {ep:2d} | Краен статус: {status} | Вкупна награда за чекорите: {ep_reward:.1f}")


env.close()


Успешно пронајден трениран Actor во локалната меморија.
 Епизода  1 | Краен статус: УСПЕШНО | Вкупна награда за чекорите: -1.0
 Епизода  2 | Краен статус: УСПЕШНО | Вкупна награда за чекорите: -2.0
 Епизода  3 | Краен статус: УСПЕШНО | Вкупна награда за чекорите: -2.0
 Епизода  4 | Краен статус: УСПЕШНО | Вкупна награда за чекорите: -2.0
 Епизода  5 | Краен статус: УСПЕШНО | Вкупна награда за чекорите: -3.0
